# Utility Placement Optimisation

**Learning outcome:** Apply utility placement optimisation through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `base`  
**Expected runtime:** under 2 minutes  
**Optional extras:** plot

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Where should two isothermal and two sensible hot and cold utility levels be placed at Process and Site hierarchy levels to minimize thermodynamic cost?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Prepare the placement study

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchWorkspace

workspace = PinchWorkspace(
    source="chocolate_factory.json", project_name="Site"
)
problem = workspace.use_case("baseline")
baseline_input = problem.to_problem_json()
search_options = {
    "iteration_limit": 1,
    "evaluation_limit": 100,
    "candidate_limit": 2,
    "run_count": 1,
}

## Step 2: Optimize a Process Zone and build its standard GCC

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
process_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    zone="Almond",
    period_ids=("0",),
    options=search_options,
)
process_evidence = process_case.utility_placement_result
process_objective = process_evidence.best.aggregate_objective
process_utilities = process_case.to_problem_json()["utilities"]
assert len(process_utilities) == 8
assert all(
    utility["name"].strip().casefold() not in {"hu", "cu"}
    for utility in process_utilities
)
process_case = workspace.add(
    process_case,
    name="optimized_process_utilities",
    activate=False,
)
process_case.target.direct_heat_integration(
    zone="Almond", period_id="0"
)
process_summary = process_case.summary_frame()
process_gcc = process_case.plot.grand_composite_curve(
    zone_name="Almond"
)

## Step 3: Optimize the Site and build its standard Total Site Profile

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
site_case = problem.target.utility_placement(
    isothermal=2,
    sensible=2,
    period_ids=("0",),
    options=search_options,
)
site_evidence = site_case.utility_placement_result
site_objective = site_evidence.best.aggregate_objective
assert len(site_case.to_problem_json()["utilities"]) == 8
site_case = workspace.add(
    site_case,
    name="optimized_site_utilities",
    activate=False,
)
assert workspace.use_case("baseline").to_problem_json() == baseline_input
site_case.target.total_site_heat_integration(period_id="0")
site_summary = site_case.summary_frame()
site_tsp = site_case.plot.total_site_profiles()

## Review the result

Review both optimized cases exactly like normal cases: compare the Process utilities on the standard GCC with the Site utilities on the standard Total Site Profile, and retain each placement result for engineering review.

In [ ]:
from IPython.display import display

display(process_objective)
display(process_summary)
display(process_gcc)
display(site_objective)
display(site_summary)
display(site_tsp)

## Interpret the result

Compare the Process result against its direct GCC and the Site result against its Total Site Profile. Inspect physical entropy generation from the balanced composite curves: use CP * ln(T_out / T_in) in kelvin for sensible intervals and the signed Q / T limit for isothermal intervals. Confirm that no generated HU/CU fallback has positive duty.

## Adapt this template

Replace the sample with validated plant data, apply defensible temperature bounds, and increase the optimizer limits before making an engineering decision.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.